# PPC v12.3 — Spine-Aware Curvature-Preserving Point Cloud Prediction

## Upgrade from v12.2 → v12.3

| Area | v12.2 | v12.3 Fix |
|---|---|---|
| Encoder | ResNet18 (11M) — shallow features | **ResNet34** — deeper, richer spine features |
| Augmentation | Brightness ±15% + flip only | **Affine + gamma + Gaussian noise + elastic + cutout** |
| N_POINTS | 5120 | **8192** — denser gap/surface coverage |
| Refinement MLP | 520→256→3 (1 hidden layer) | **520→384→384→3** (2 hidden layers, wider) |
| Losses | 14 losses — 5 redundant gap losses | **8 losses** — deduplicated, + extent_match + curvature |
| Compactness fix | None — predictions too compact | **extent_match_loss** forces per-axis spread to match GT |
| Curvature | Not modeled | **curvature_loss** — local angular consistency along spine Z |
| LR schedule | CosineWarmRestarts (restart@50 destroys progress) | **OneCycleLR** — single sweep, no restarts |
| Batch size | 2 (noisy gradients) | **2 × 4 accumulation = effective 8** |
| Chamfer truncation | 15 mm (too generous) | **8 mm** — sharper signal on near-surface points |
| Inference | Single pass | **TTA** — original + H-flip averaged |

In [ ]:
"""
PPC v12.3 — Spine-Aware, Curvature-Preserving, Anti-Compactness
================================================================
Key: ResNet34 encoder, 8192 points, wider MLP, streamlined losses,
     extent matching (anti-compactness), curvature loss, strong augmentation,
     OneCycleLR, gradient accumulation, TTA at test.
"""
import os, sys, json, time, random, warnings, csv, math
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image, ImageFilter
import vtk
from tqdm import tqdm
from scipy.ndimage import gaussian_filter, map_coordinates

warnings.filterwarnings("ignore", category=FutureWarning)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {device}")
if torch.cuda.is_available():
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    total_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    torch.cuda.set_per_process_memory_fraction(min(50.0 / total_gb, 0.95), 0)

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True

# ── PATHS ─────────────────────────────────────────────────────────────────────
DATA_ROOT   = Path("/apps/app/see_all_ai/dataset/Lumbar_Filtered_1037")
SPLIT_FILE  = DATA_ROOT / "dataset_split.json"
PROJECT_DIR = Path("/apps/app/pandu/ppc_network_v12_3")
CKPT_DIR    = PROJECT_DIR / "checkpoints"
LOG_DIR     = PROJECT_DIR / "logs"
RESULTS_DIR = PROJECT_DIR / "results"
for d in [PROJECT_DIR, CKPT_DIR, LOG_DIR, RESULTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── MODEL CONFIG ──────────────────────────────────────────────────────────────
IMG_SIZE            = 512
COARSE_VOL_SIZE     = 32
AUX_VOL_SIZE        = 48
GAP_VOL_SIZE        = 160
N_POINTS            = 8192          # ← UP from 5120: denser coverage
ENC_CHANNELS        = 192
VOL_CHANNELS        = 128
DEC_CHANNELS        = 128
QUERY_DIM           = 384           # ← UP from 256: wider refinement MLP
N_REFINE_STAGES     = 3
PRETRAINED_IMAGENET = True

# ── TRAINING CONFIG ───────────────────────────────────────────────────────────
BATCH_SIZE          = 2
ACCUM_STEPS         = 4             # ← NEW: effective batch = 8
NUM_WORKERS         = 4
EPOCHS              = 300
LR                  = 3e-4          # ← UP: higher peak LR for OneCycleLR
WEIGHT_DECAY        = 5e-5          # ← DOWN: less aggressive
WARMUP_PCT          = 0.05          # 5% of total steps for OneCycleLR warmup
USE_AMP             = False
GRAD_CLIP           = 1.0
EARLY_STOP_PATIENCE = 60
FREEZE_ENC_EPOCHS   = 5

# ── LOSS WEIGHTS (streamlined: 8 losses, no redundancy) ──────────────────────
# Phase 1 (ep 1+): Core geometry
W_CHAMFER   = 1.0
W_GAP       = 4.0                   # gap voxel penalty
W_AXIAL     = 0.8                   # Z-density match
W_AUX_OCC   = 0.05                  # auxiliary occupancy
W_EXTENT    = 2.0                   # ← NEW: anti-compactness, match GT spread

# Phase 2 (ep 5+): Structure + curvature
W_SW        = 0.3                   # sliced Wasserstein
W_PROJ      = 0.02                  # 2D projection Chamfer
W_CURVE     = 0.5                   # ← NEW: local spine curvature consistency

PHASE2_EPOCH = 5

MAX_EVAL     = 103
CKPT_PATH    = CKPT_DIR / "latest_checkpoint.pth"
BEST_CKPT    = CKPT_DIR / "best_checkpoint.pth"
TRAINING_LOG = LOG_DIR  / "training_log.json"

print("=" * 65)
print("PPC v12.3 — Spine-Aware, Anti-Compactness, Curvature-Preserving")
print("=" * 65)
for k, v in dict(
    ENCODER="ResNet34 (upgrade from ResNet18)",
    N_POINTS=N_POINTS, QUERY_DIM=QUERY_DIM,
    EFFECTIVE_BATCH=f"{BATCH_SIZE}×{ACCUM_STEPS}={BATCH_SIZE*ACCUM_STEPS}",
    LR=f"{LR} (OneCycleLR)", GRAD_CLIP=GRAD_CLIP,
    LOSSES="8 (chamfer+gap+axial+aux+extent | sw+proj+curvature)",
    PHASE1="ep 1+: chamfer+gap+axial+aux_occ+extent",
    PHASE2=f"ep {PHASE2_EPOCH}+: +sw+proj+curvature",
    PATIENCE=EARLY_STOP_PATIENCE, EPOCHS=EPOCHS,
    AUGMENTATION="affine+gamma+noise+elastic+cutout+flip",
).items():
    print(f"  {k:<18} = {v}")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# DATA UTILITIES — v12.3: spine-aware augmentation
# ══════════════════════════════════════════════════════════════════════════════

def load_vtk_points(path):
    r = vtk.vtkPolyDataReader(); r.SetFileName(str(path)); r.Update()
    p = r.GetOutput()
    return np.array([p.GetPoint(i) for i in range(p.GetNumberOfPoints())], np.float32)

def save_vtk_points(points, path):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    vp = vtk.vtkPoints()
    for pt in points: vp.InsertNextPoint(float(pt[0]), float(pt[1]), float(pt[2]))
    vc = vtk.vtkCellArray()
    for i in range(len(points)): vc.InsertNextCell(1); vc.InsertCellPoint(i)
    pd = vtk.vtkPolyData(); pd.SetPoints(vp); pd.SetVerts(vc)
    w = vtk.vtkPolyDataWriter()
    w.SetFileName(str(path)); w.SetInputData(pd); w.SetFileTypeToASCII(); w.Write()

def load_drr_image(path, size=IMG_SIZE):
    img = Image.open(path).convert("L")
    if img.size != (size, size): img = img.resize((size, size), Image.BILINEAR)
    return np.array(img, np.float32) / 255.0

def load_projection_matrix(path):
    with open(path) as f: vals = [float(v) for v in f.read().split()]
    return np.array(vals, np.float32).reshape(3, 4)

def load_split(p=SPLIT_FILE):
    with open(p) as f: d = json.load(f)
    return d["train"], d["val"], d["test"]

def append_log(path, rec):
    log = {"records": []}
    if path.exists():
        with open(path) as f: log = json.load(f)
    log["records"].append(rec)
    tmp = path.with_suffix(".tmp")
    with open(tmp, "w") as f: json.dump(log, f, indent=2)
    tmp.replace(path)

def pts_to_local(pts, c, s): return (pts - c[None]) / (s[None] + 1e-6)
def local_to_world(pts, c, s): return pts * s[:, None, :] + c[:, None, :]

def compute_scale(gt):
    e = (gt.max(0) - gt.min(0)).astype(np.float32)
    s = e * 0.55 + np.array([20., 20., 30.], np.float32)
    return np.maximum(s, np.array([50., 50., 80.], np.float32))

def make_aux_occ(pl, vs=AUX_VOL_SIZE, d=1):
    p   = np.clip((np.asarray(pl, np.float32) + 1) * 0.5, 0, 0.999999)
    idx = np.clip(np.floor(p * vs).astype(np.int32), 0, vs - 1)
    o   = np.zeros((vs,) * 3, np.float32)
    for dx in range(-d, d + 1):
        for dy in range(-d, d + 1):
            for dz in range(-d, d + 1):
                o[np.clip(idx[:,0]+dx,0,vs-1),
                  np.clip(idx[:,1]+dy,0,vs-1),
                  np.clip(idx[:,2]+dz,0,vs-1)] = 1.0
    return o

def make_gap_occ(pl, vs=GAP_VOL_SIZE):
    p   = np.clip((np.asarray(pl, np.float32) + 1) * 0.5, 0, 0.999999)
    idx = np.clip(np.floor(p * vs).astype(np.int32), 0, vs - 1)
    o   = np.zeros((vs,) * 3, np.float32)
    o[idx[:,0], idx[:,1], idx[:,2]] = 1.0
    return o

def flip_P(P, s=IMG_SIZE):
    return np.array([[-1,0,s-1],[0,1,0],[0,0,1]], np.float32) @ P


# ── AUGMENTATION FUNCTIONS (v12.3 — much stronger) ───────────────────────────

def aug_gamma(img, lo=0.7, hi=1.5):
    """Random gamma correction — better than linear brightness."""
    gamma = np.random.uniform(lo, hi)
    return np.clip(np.power(img, gamma), 0, 1)

def aug_gaussian_noise(img, std_range=(0.0, 0.03)):
    """Additive Gaussian noise."""
    std = np.random.uniform(*std_range)
    return np.clip(img + np.random.randn(*img.shape).astype(np.float32) * std, 0, 1)

def aug_elastic(img, alpha=8.0, sigma=3.0):
    """Light elastic deformation — simulates soft-tissue variability in DRR."""
    h, w = img.shape
    dx = gaussian_filter(np.random.randn(h, w).astype(np.float32), sigma) * alpha
    dy = gaussian_filter(np.random.randn(h, w).astype(np.float32), sigma) * alpha
    y, x = np.meshgrid(np.arange(h), np.arange(w), indexing='ij')
    coords = [np.clip(y + dy, 0, h - 1), np.clip(x + dx, 0, w - 1)]
    return map_coordinates(img, coords, order=1, mode='reflect').astype(np.float32)

def aug_cutout(img, n_holes=2, max_sz=40):
    """Random rectangular occlusions — simulates artifact/overlap."""
    out = img.copy()
    h, w = out.shape
    for _ in range(n_holes):
        sz_h = np.random.randint(10, max_sz)
        sz_w = np.random.randint(10, max_sz)
        cy, cx = np.random.randint(0, h), np.random.randint(0, w)
        y1, y2 = max(0, cy - sz_h//2), min(h, cy + sz_h//2)
        x1, x2 = max(0, cx - sz_w//2), min(w, cx + sz_w//2)
        out[y1:y2, x1:x2] = 0.0
    return out

def aug_affine_drr(img, P, max_rot_deg=3.0, max_trans_pix=8, max_scale=0.04):
    """
    Small affine perturbation of 2D DRR + matching adjustment to projection matrix.
    Simulates slight patient positioning variation.
    """
    h, w = img.shape
    angle = np.random.uniform(-max_rot_deg, max_rot_deg) * np.pi / 180.0
    tx = np.random.uniform(-max_trans_pix, max_trans_pix)
    ty = np.random.uniform(-max_trans_pix, max_trans_pix)
    sc = 1.0 + np.random.uniform(-max_scale, max_scale)

    ca, sa = np.cos(angle), np.sin(angle)
    cx, cy = w / 2.0, h / 2.0

    # Affine: translate to center → rotate+scale → translate back + shift
    M = np.array([
        [sc * ca, -sc * sa, (1 - sc*ca)*cx + sc*sa*cy + tx],
        [sc * sa,  sc * ca, -sc*sa*cx + (1 - sc*ca)*cy + ty],
    ], dtype=np.float32)

    # Apply to image
    from PIL import Image as _Im
    pil_img = _Im.fromarray((img * 255).astype(np.uint8), mode='L')
    M_pil = [M[0,0], M[0,1], M[0,2], M[1,0], M[1,1], M[1,2]]
    pil_out = pil_img.transform((w, h), _Im.AFFINE, M_pil,
                                 resample=_Im.BILINEAR, fillcolor=0)
    img_out = np.array(pil_out, np.float32) / 255.0

    # Adjust projection matrix: P_new = M_3x3 @ P
    M3 = np.eye(3, dtype=np.float32)
    M3[:2, :] = M[:2, :]  # embed 2×3 affine into 3×3
    P_out = M3 @ P

    return img_out, P_out


class LumbarDataset(Dataset):
    def __init__(self, ids, root=DATA_ROOT, aug=False):
        self.ids = ids; self.root = Path(root); self.aug = aug
        self.norm = transforms.Normalize(mean=[0.485], std=[0.229])

    def __len__(self): return len(self.ids)

    def __getitem__(self, i):
        pid = self.ids[i]; d = self.root / pid
        dap = load_drr_image(d / "AP_0"  / "drr_AP_0.png")
        dlp = load_drr_image(d / "LP_90" / "drr_LP_90.png")
        Pap = load_projection_matrix(d / "AP_0"  / "P_AP_0.txt")
        Plp = load_projection_matrix(d / "LP_90" / "P_LP_90.txt")
        gt  = load_vtk_points(d / "gt_ppc.vtk")
        c   = gt.mean(0).astype(np.float32); s = compute_scale(gt)
        gl  = np.clip(pts_to_local(gt, c, s), -1, 1)
        ao  = make_aux_occ(gl); go = make_gap_occ(gl)
        n   = len(gt)
        sel = np.random.choice(n, N_POINTS, replace=(n < N_POINTS))

        if self.aug:
            # 1) Random horizontal flip (50%)
            if np.random.rand() < 0.5:
                dap = dap[:, ::-1].copy(); dlp = dlp[:, ::-1].copy()
                Pap = flip_P(Pap); Plp = flip_P(Plp)

            # 2) Small affine perturbation (70%)
            if np.random.rand() < 0.7:
                dap, Pap = aug_affine_drr(dap, Pap, max_rot_deg=3.0,
                                           max_trans_pix=6, max_scale=0.03)
                dlp, Plp = aug_affine_drr(dlp, Plp, max_rot_deg=3.0,
                                           max_trans_pix=6, max_scale=0.03)

            # 3) Gamma correction (80%)
            if np.random.rand() < 0.8:
                dap = aug_gamma(dap, 0.7, 1.4)
                dlp = aug_gamma(dlp, 0.7, 1.4)

            # 4) Gaussian noise (60%)
            if np.random.rand() < 0.6:
                dap = aug_gaussian_noise(dap, (0.0, 0.025))
                dlp = aug_gaussian_noise(dlp, (0.0, 0.025))

            # 5) Light elastic deformation (30%)
            if np.random.rand() < 0.3:
                dap = aug_elastic(dap, alpha=6.0, sigma=3.0)
                dlp = aug_elastic(dlp, alpha=6.0, sigma=3.0)

            # 6) Cutout (30%)
            if np.random.rand() < 0.3:
                dap = aug_cutout(dap, n_holes=1, max_sz=30)
                dlp = aug_cutout(dlp, n_holes=1, max_sz=30)

        return {
            "drr_ap":       self.norm(torch.from_numpy(dap).unsqueeze(0).float()),
            "drr_lp":       self.norm(torch.from_numpy(dlp).unsqueeze(0).float()),
            "P_ap":         torch.from_numpy(Pap.copy()).float(),
            "P_lp":         torch.from_numpy(Plp.copy()).float(),
            "gt_ppc_world": torch.from_numpy(gt[sel]).float(),
            "gt_ppc_local": torch.from_numpy(gl[sel]).float(),
            "aux_occ":      torch.from_numpy(ao).float(),
            "gap_occ":      torch.from_numpy(go).float(),
            "center":       torch.from_numpy(c).float(),
            "scale":        torch.from_numpy(s).float(),
            "patient_id":   pid,
        }

train_ids, val_ids, test_ids = load_split()
print(f"Split: train={len(train_ids)}  val={len(val_ids)}  test={len(test_ids)}")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# MODEL — v12.3: ResNet34 encoder, wider refinement MLP
# ══════════════════════════════════════════════════════════════════════════════

class Encoder2D(nn.Module):
    """v12.3: ResNet34 backbone — deeper features for fine spine detail."""
    def __init__(self, in_ch=1, out_ch=ENC_CHANNELS, pretrained=PRETRAINED_IMAGENET):
        super().__init__()
        base = models.resnet34(
            weights=models.ResNet34_Weights.IMAGENET1K_V1 if pretrained else None)
        c1 = nn.Conv2d(in_ch, 64, 7, stride=2, padding=3, bias=False)
        if pretrained:
            with torch.no_grad():
                c1.weight[:] = base.conv1.weight.mean(1, keepdim=True)
        base.conv1 = c1
        self.stem   = nn.Sequential(base.conv1, base.bn1, base.relu, base.maxpool)
        self.layer1 = base.layer1
        self.layer2 = base.layer2
        self.layer3 = base.layer3
        self.proj   = nn.Conv2d(256, out_ch, 1)

    def forward(self, x):
        return self.proj(self.layer3(self.layer2(self.layer1(self.stem(x)))))


class FeatureLift(nn.Module):
    def __init__(self, in_ch=ENC_CHANNELS, out_ch=VOL_CHANNELS, depth=COARSE_VOL_SIZE):
        super().__init__()
        self.depth_embed = nn.Parameter(torch.randn(1, in_ch, depth, 1, 1) * 0.02)
        self.refine = nn.Sequential(
            nn.Conv3d(in_ch, out_ch, 3, padding=1), nn.GroupNorm(8, out_ch), nn.GELU(),
            nn.Conv3d(out_ch, out_ch, 3, padding=1), nn.GroupNorm(8, out_ch), nn.GELU())

    def forward(self, f2d):
        B, C, H, W = f2d.shape
        vol = f2d.unsqueeze(2).expand(B, C, COARSE_VOL_SIZE, H, W).contiguous()
        return self.refine(vol + self.depth_embed)


class BiplanarFusion(nn.Module):
    def __init__(self, in_ch=VOL_CHANNELS, out_ch=VOL_CHANNELS):
        super().__init__()
        self.fuse = nn.Sequential(
            nn.Conv3d(in_ch*2, out_ch, 1), nn.GroupNorm(8, out_ch), nn.GELU(),
            nn.Conv3d(out_ch, out_ch, 3, padding=1), nn.GroupNorm(8, out_ch), nn.GELU())

    def forward(self, ap, lp):
        return self.fuse(torch.cat([
            ap.permute(0,1,4,2,3).contiguous(),
            lp.permute(0,1,2,4,3).contiguous()], 1))


class Block3D(nn.Module):
    def __init__(self, ic, oc):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv3d(ic, oc, 3, padding=1), nn.GroupNorm(8, oc), nn.GELU(),
            nn.Conv3d(oc, oc, 3, padding=1), nn.GroupNorm(8, oc), nn.GELU())

    def forward(self, x): return self.block(x)


class CoarseUNet3D(nn.Module):
    def __init__(self, in_ch=VOL_CHANNELS, fc=DEC_CHANNELS):
        super().__init__()
        self.enc1 = Block3D(in_ch, 96); self.down1 = nn.Conv3d(96, 160, 2, stride=2)
        self.enc2 = Block3D(160, 160);  self.down2 = nn.Conv3d(160, 224, 2, stride=2)
        self.bottleneck = Block3D(224, 224)
        self.up2  = nn.ConvTranspose3d(224, 160, 2, stride=2); self.dec2 = Block3D(320, 160)
        self.up1  = nn.ConvTranspose3d(160,  96, 2, stride=2); self.dec1 = Block3D(192, fc)
        self.aux_head = nn.Sequential(
            nn.Conv3d(fc, fc//2, 3, padding=1), nn.GELU(), nn.Conv3d(fc//2, 1, 1))

    def forward(self, x):
        e1 = self.enc1(x); e2 = self.enc2(self.down1(e1))
        b  = self.bottleneck(self.down2(e2))
        d2 = self.dec2(torch.cat([self.up2(b), e2], 1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], 1))
        aux = F.interpolate(self.aux_head(d1), size=(AUX_VOL_SIZE,)*3,
                            mode="trilinear", align_corners=True)
        return d1, aux


def project_points(P, pts, s=IMG_SIZE):
    B, N, _ = pts.shape
    h   = torch.cat([pts, torch.ones(B, N, 1, device=pts.device, dtype=pts.dtype)], -1)
    uvw = torch.bmm(h, P.transpose(1, 2)); z = uvw[..., 2:3].clamp(min=1e-6)
    uv  = uvw[..., :2] / z
    return uv, (uv / (s-1)) * 2 - 1, torch.log(z)

def sample_2d(fm, uvn):
    return F.grid_sample(fm, uvn.view(fm.shape[0], -1, 1, 2), mode="bilinear",
                         align_corners=True, padding_mode="border").squeeze(-1).transpose(1, 2)

def sample_3d(vf, pl):
    g = torch.stack([pl[...,2], pl[...,1], pl[...,0]], -1).view(pl.shape[0], -1, 1, 1, 3)
    return F.grid_sample(vf, g, mode="bilinear", align_corners=True,
                         padding_mode="border").squeeze(-1).squeeze(-1).transpose(1, 2)


class QueryInit(nn.Module):
    """v12.3: denser base grid → 8192 points."""
    def __init__(self, n=N_POINTS, fc=DEC_CHANNELS):
        super().__init__()
        # 20×20×20 = 8000 base, padded to N_POINTS
        g = np.stack(np.meshgrid(
            np.linspace(-.85, .85, 20),
            np.linspace(-.85, .85, 20),
            np.linspace(-.92, .92, 21), indexing="ij"), -1
        ).reshape(-1, 3).astype(np.float32)
        # Subsample or pad to exactly n
        if len(g) >= n:
            idx = np.random.choice(len(g), n, replace=False)
            g = g[idx]
        else:
            extra = n - len(g)
            pad = g[np.random.choice(len(g), extra, replace=True)]
            pad += np.random.randn(extra, 3).astype(np.float32) * 0.02
            g = np.concatenate([g, pad], 0)
        self.register_buffer("base", torch.from_numpy(g)); self.n = n
        self.head = nn.Sequential(nn.AdaptiveAvgPool3d(1), nn.Flatten(),
                                  nn.Linear(fc, 384), nn.GELU(), nn.Linear(384, n*3))

    def forward(self, vf, aux=None):
        B = vf.shape[0]; off = self.head(vf).view(B, self.n, 3)
        raw = self.base[None] + 0.20 * torch.tanh(off)
        if aux is not None:
            gs   = torch.stack([raw[...,2], raw[...,1], raw[...,0]], -1).view(B, self.n, 1, 1, 3)
            gate = torch.sigmoid(F.grid_sample(aux, gs, mode="bilinear", align_corners=True,
                                               padding_mode="border").view(B, self.n)).unsqueeze(-1)
            raw  = self.base[None] + gate * 0.25 * torch.tanh(off)
        return torch.clamp(raw, -1, 1)


class RefinementStage(nn.Module):
    """v12.3: wider 2-hidden-layer MLP (520 → 384 → 384 → 3)."""
    def __init__(self, f2d=ENC_CHANNELS, f3d=DEC_CHANNELS, h=QUERY_DIM):
        super().__init__()
        in_dim = f2d*2 + f3d + 3 + 2 + 2 + 1  # = 520
        self.mlp = nn.Sequential(
            nn.Linear(in_dim, h), nn.GELU(), nn.Dropout(0.05),
            nn.Linear(h, h), nn.GELU(), nn.Dropout(0.05),
            nn.Linear(h, 3))

    def forward(self, q, vf, fap, flp, Pap, Plp, c, s, aux=None):
        qw = local_to_world(q, c, s)
        _, ua, _ = project_points(Pap, qw)
        _, ul, _ = project_points(Plp, qw)
        B, N, _ = q.shape
        if aux is not None:
            gs = torch.stack([q[...,2], q[...,1], q[...,0]], -1).view(B, N, 1, 1, 3)
            ov = F.grid_sample(aux, gs, mode="bilinear", align_corners=True,
                               padding_mode="border").view(B, N, 1)
        else:
            ov = torch.zeros(B, N, 1, device=q.device)
        x     = torch.cat([sample_3d(vf, q), sample_2d(fap, ua),
                           sample_2d(flp, ul), q, ua, ul, ov], -1)
        delta = 0.25 * torch.tanh(self.mlp(x))
        return q + delta, {"delta": delta}


class PPCNetV12(nn.Module):
    """v12.3: ResNet34, wider MLP, 8192 points."""
    def __init__(self):
        super().__init__()
        self.encoder_ap = Encoder2D(); self.encoder_lp = Encoder2D()
        self.lift_ap    = FeatureLift(); self.lift_lp  = FeatureLift()
        self.fusion     = BiplanarFusion(); self.coarse3d = CoarseUNet3D()
        self.query_init = QueryInit()
        self.stages     = nn.ModuleList([RefinementStage() for _ in range(N_REFINE_STAGES)])

    def forward(self, dap, dlp, Pap, Plp, c, s):
        fap = self.encoder_ap(dap); flp = self.encoder_lp(dlp)
        vf, aux = self.coarse3d(self.fusion(self.lift_ap(fap), self.lift_lp(flp)))
        q = self.query_init(vf, aux); sa = []
        for stage in self.stages:
            q, a = stage(q, vf, fap, flp, Pap, Plp, c, s, aux); sa.append(a)
        return {"pred_local": torch.clamp(q, -1, 1),
                "pred_world": local_to_world(q, c, s),
                "aux_occ_logits": aux, "stage_aux": sa}


def count_params(m): return sum(p.numel() for p in m.parameters() if p.requires_grad)
_m = PPCNetV12(); print(f"PPCNetV12 (v12.3) params: {count_params(_m)/1e6:.1f} M"); del _m


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# LOSS FUNCTIONS — v12.3: 8 focused losses, anti-compactness, curvature
# ══════════════════════════════════════════════════════════════════════════════
#
# v12.2 had 14 losses with 5 redundant gap losses fighting each other.
# v12.3 keeps the best-performing subset and adds two spine-specific losses:
#   - extent_match_loss: prevents compactness by matching per-axis spread
#   - curvature_loss: preserves local angular consistency along spine Z-axis
#

def safe_loss(val, name='loss', cap=1e4):
    if torch.isnan(val) or torch.isinf(val):
        print(f"  ⚠ NaN/Inf in {name}, returning 0")
        return torch.tensor(0.0, device=val.device)
    return torch.clamp(val, max=cap)


def pairwise_sq(x, y):
    x2 = (x ** 2).sum(-1, keepdim=True)
    y2 = (y ** 2).sum(-1).unsqueeze(1)
    return (x2 + y2 - 2.0 * torch.bmm(x, y.transpose(1, 2))).clamp_min(0)


# ── Phase 1: Core geometry (ep 1+) ───────────────────────────────────────────

def chamfer_loss(pred, gt, wp=1.0, wg=1.5, tr=8.0):
    """
    Bidirectional Chamfer — v12.3: truncation lowered from 15→8 mm.
    Sharper gradient on near-surface points that matter at sub-2mm scale.
    """
    d2 = pairwise_sq(pred, gt)
    fwd = torch.clamp(d2.min(2)[0], max=tr ** 2).mean()
    bwd = torch.clamp(d2.min(1)[0], max=tr ** 2).mean()
    return wp * fwd + wg * bwd


def gap_penalty(pl, gap_occ):
    """Penalize points in empty GT voxels. Output in [0, 1]."""
    B, N, _ = pl.shape
    occ = gap_occ.unsqueeze(1)
    g = torch.stack([pl[..., 2], pl[..., 1], pl[..., 0]], -1).view(B, N, 1, 1, 3)
    s = F.grid_sample(occ, g, mode="bilinear", align_corners=True,
                      padding_mode="border").view(B, N)
    return ((1.0 - s).clamp(min=0) ** 2).mean()


def axial_density(pl, gl, nb=80, sig=0.020):
    """Match Z-axis density distribution via KDE."""
    c  = torch.linspace(-1, 1, nb, device=pl.device)
    pk = torch.exp(-0.5 * ((pl[..., 2].unsqueeze(-1) - c) / sig) ** 2).sum(1)
    gk = torch.exp(-0.5 * ((gl[..., 2].unsqueeze(-1) - c) / sig) ** 2).sum(1)
    pd = pk / (pk.sum(1, keepdim=True) + 1e-2)
    gd = gk / (gk.sum(1, keepdim=True) + 1e-2)
    return (pd - gd).abs().sum(1).mean()


def dice_loss(logits, tgt, eps=1e-6):
    p = torch.sigmoid(logits).reshape(logits.shape[0], -1)
    t = tgt.reshape(tgt.shape[0], -1)
    return 1.0 - ((2 * (p * t).sum(1) + eps) / (p.sum(1) + t.sum(1) + eps)).mean()


def extent_match_loss(pred_local, gt_local):
    """
    ★ NEW in v12.3 — Anti-compactness loss.

    Problem: v12.2 predictions are systematically more compact than GT.
    The model "plays it safe" by clustering points toward the center,
    because Chamfer loss is asymmetric — missing an outlier costs less
    than placing a point far from any GT.

    Fix: explicitly match the per-axis extent (spread) of pred vs GT.
    Uses percentile-based spread (5th to 95th) rather than min/max
    to be robust to outliers while still pushing the point cloud to
    cover the full GT envelope.

    Also matches per-axis std to ensure interior density spread matches.
    """
    B = pred_local.shape[0]
    loss = torch.tensor(0.0, device=pred_local.device)

    for axis in range(3):
        pz = pred_local[..., axis]  # (B, N)
        gz = gt_local[..., axis]    # (B, N)

        # Percentile-based extent (differentiable via sorting)
        ps = pz.sort(dim=1)[0]
        gs = gz.sort(dim=1)[0]
        Np, Ng = ps.shape[1], gs.shape[1]

        # 5th and 95th percentile indices
        p_lo = ps[:, max(0, int(0.05 * Np))]
        p_hi = ps[:, min(Np-1, int(0.95 * Np))]
        g_lo = gs[:, max(0, int(0.05 * Ng))]
        g_hi = gs[:, min(Ng-1, int(0.95 * Ng))]

        p_extent = p_hi - p_lo
        g_extent = g_hi - g_lo

        # Penalize pred being more compact than GT (asymmetric — allow slight over-spread)
        extent_diff = g_extent - p_extent
        loss = loss + F.relu(extent_diff).mean()  # only penalize under-spread

        # Also match std (overall spread shape)
        p_std = pz.std(dim=1)
        g_std = gz.std(dim=1)
        loss = loss + (p_std - g_std).abs().mean() * 0.5

    return loss / 3.0


# ── Phase 2: Structure + curvature (ep PHASE2_EPOCH+) ────────────────────────

def sw_loss(pred, gt, np_=50):
    """Sliced Wasserstein distance."""
    dirs = F.normalize(torch.randn(np_, 3, device=pred.device), -1)
    pp = torch.einsum("bnd,pd->bnp", pred, dirs)
    gp = torch.einsum("bnd,pd->bnp", gt, dirs)
    return ((pp.sort(1)[0] - gp.sort(1)[0]) ** 2).mean()


def proj_loss(pw, gw, Pap, Plp):
    """2D projection Chamfer — normalized by IMG_SIZE²."""
    def ch2d(a, b):
        d2 = pairwise_sq(a, b) / (IMG_SIZE ** 2 + 1e-6)
        return 0.5 * (torch.clamp(d2.min(2)[0], max=1.0).mean()
                    + torch.clamp(d2.min(1)[0], max=1.0).mean())
    pa, _, _ = project_points(Pap, pw); ga, _, _ = project_points(Pap, gw)
    pl, _, _ = project_points(Plp, pw); gl, _, _ = project_points(Plp, gw)
    return ch2d(pa, ga) + ch2d(pl, gl)


def curvature_loss(pred_local, gt_local, n_slices=20, k_ring=3):
    """
    ★ NEW in v12.3 — Spine curvature consistency loss.

    The lumbar spine has a characteristic lordotic curve. This loss ensures
    the predicted point cloud preserves local angular consistency along
    the Z-axis (cranio-caudal direction).

    Method: Slice both pred and GT along Z into n_slices bands.
    For each band, compute the centroid (center of mass in XY).
    The sequence of centroids traces the spine's curvature.
    Loss = L1 difference between pred and GT centroid trajectories,
    plus angular consistency (second derivative smoothness).

    This is stable: just means and differences, no eigendecomposition.
    """
    B = pred_local.shape[0]
    device = pred_local.device

    z_edges = torch.linspace(-1, 1, n_slices + 1, device=device)
    z_centers = 0.5 * (z_edges[:-1] + z_edges[1:])

    total_loss = torch.tensor(0.0, device=device)

    for b in range(B):
        pred_centroids = []
        gt_centroids = []
        valid_slices = []

        for si in range(n_slices):
            z_lo, z_hi = z_edges[si], z_edges[si + 1]

            # Soft assignment: Gaussian weighting centered on slice
            sig_slice = (z_hi - z_lo) * 0.6  # slight overlap between slices
            pred_w = torch.exp(-0.5 * ((pred_local[b, :, 2] - z_centers[si]) / sig_slice) ** 2)
            gt_w   = torch.exp(-0.5 * ((gt_local[b, :, 2] - z_centers[si]) / sig_slice) ** 2)

            pw_sum = pred_w.sum() + 1e-6
            gw_sum = gt_w.sum() + 1e-6

            # Only use slices where both pred and GT have reasonable density
            if pw_sum > 1.0 and gw_sum > 1.0:
                p_centroid = (pred_w.unsqueeze(-1) * pred_local[b, :, :2]).sum(0) / pw_sum
                g_centroid = (gt_w.unsqueeze(-1) * gt_local[b, :, :2]).sum(0) / gw_sum
                pred_centroids.append(p_centroid)
                gt_centroids.append(g_centroid)
                valid_slices.append(si)

        if len(valid_slices) >= 3:
            pc = torch.stack(pred_centroids)  # (S, 2)
            gc = torch.stack(gt_centroids)    # (S, 2)

            # 1) Centroid trajectory matching (L1)
            traj_loss = (pc - gc).abs().mean()

            # 2) Angular consistency: compare second derivatives (curvature proxy)
            # First derivative (tangent direction)
            p_d1 = pc[1:] - pc[:-1]
            g_d1 = gc[1:] - gc[:-1]
            # Second derivative (curvature)
            if len(p_d1) >= 2:
                p_d2 = p_d1[1:] - p_d1[:-1]
                g_d2 = g_d1[1:] - g_d1[:-1]
                curv_loss = (p_d2 - g_d2).abs().mean()
                total_loss = total_loss + traj_loss + 0.5 * curv_loss
            else:
                total_loss = total_loss + traj_loss

    return total_loss / max(B, 1)


# ── Combined loss with 2-phase ramp-in ───────────────────────────────────────

def total_loss(out, batch, epoch=0):
    """
    v12.3 combined loss — 8 losses in 2 phases (down from 14 in 3 phases).
    Phase 1 (ep 1+):  chamfer + gap + axial + aux_occ + extent
    Phase 2 (ep 5+):  + sw + proj + curvature
    """
    pw, pl = out["pred_world"], out["pred_local"]
    gw, gl = batch["gt_ppc_world"], batch["gt_ppc_local"]

    # ── Phase 1: Core geometry (always active) ──
    lc   = safe_loss(chamfer_loss(pw, gw), "chamfer")
    lg   = safe_loss(gap_penalty(pl, batch["gap_occ"]), "gap")
    la   = safe_loss(axial_density(pl, gl), "axial")
    laux = safe_loss(
        F.binary_cross_entropy_with_logits(
            out["aux_occ_logits"].squeeze(1), batch["aux_occ"])
        + dice_loss(out["aux_occ_logits"].squeeze(1), batch["aux_occ"]),
        "aux_occ")
    lext = safe_loss(extent_match_loss(pl, gl), "extent")

    loss = (W_CHAMFER * lc + W_GAP * lg + W_AXIAL * la
            + W_AUX_OCC * laux + W_EXTENT * lext)

    bd = {"chamfer": lc, "gap": lg, "axial": la, "aux_occ": laux, "extent": lext}

    # ── Phase 2: Structure + curvature (ramped in from PHASE2_EPOCH) ──
    if epoch >= PHASE2_EPOCH:
        ramp2 = min(1.0, (epoch - PHASE2_EPOCH + 1) / 5.0)
        ls   = safe_loss(sw_loss(pl, gl), "sw")
        lp   = safe_loss(proj_loss(pw, gw, batch["P_ap"], batch["P_lp"]), "proj")
        lcur = safe_loss(curvature_loss(pl, gl), "curvature")

        loss = loss + ramp2 * (W_SW * ls + W_PROJ * lp + W_CURVE * lcur)
        bd.update({"sw": ls, "proj": lp, "curvature": lcur})

    bd_f = {"total": float(loss.detach())}
    for k, v in bd.items():
        bd_f[k] = float(v.detach()) if isinstance(v, torch.Tensor) else float(v)

    return loss, bd_f


print("v12.3 loss functions defined (8 losses, 2 phases).")
print(f"  Phase 1 (ep 1+):  chamfer, gap, axial, aux_occ, extent")
print(f"  Phase 2 (ep {PHASE2_EPOCH}+):  + sw, proj, curvature")
print(f"  Removed: zgap, offset, smooth, spacing, boundary, ivg, vsep, gwidth")
print(f"  Added:   extent_match (anti-compactness), curvature (spine shape)")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# TRAINING LOOP — v12.3: OneCycleLR, gradient accumulation, clean
# ══════════════════════════════════════════════════════════════════════════════

def collate_fn(b):
    out = {}
    for k in b[0]:
        vals = [x[k] for x in b]
        out[k] = torch.stack(vals, 0) if isinstance(vals[0], torch.Tensor) else vals
    return out

def to_dev(b):
    return {k: (v.to(device, non_blocking=True) if isinstance(v, torch.Tensor) else v)
            for k, v in b.items()}

def chamfer_np(pred, gt):
    pred, gt = np.asarray(pred, np.float32), np.asarray(gt, np.float32)
    d = np.linalg.norm(pred[:, None] - gt[None], axis=-1)
    dp, dg = d.min(1), d.min(0)
    def fs(th): p, r = (dp<=th).mean(), (dg<=th).mean(); return 2*p*r/(p+r) if p+r>0 else 0.
    return {"chamfer_mm": float(0.5*(dp.mean()+dg.mean())),
            "fscore_1mm": float(fs(1)), "fscore_2mm": float(fs(2)),
            "fscore_5mm": float(fs(5)),
            "hausdorff_95": float(max(np.percentile(dp,95), np.percentile(dg,95)))}


# ── Build model ───────────────────────────────────────────────────────────────
model = PPCNetV12().to(device)
print(f"Params : {count_params(model)/1e6:.1f} M")
print("Training from SCRATCH — ImageNet encoder (ResNet34).")

enc_params   = list(model.encoder_ap.parameters()) + list(model.encoder_lp.parameters())
enc_ids      = {id(p) for p in enc_params}
other_params = [p for p in model.parameters() if id(p) not in enc_ids]

optimizer = torch.optim.AdamW([
    {"params": enc_params,   "lr": LR * 0.3},   # v12.3: higher encoder LR (0.3x vs 0.1x)
    {"params": other_params, "lr": LR},
], weight_decay=WEIGHT_DECAY)

train_ds     = LumbarDataset(train_ids, aug=True)
val_ds       = LumbarDataset(val_ids,   aug=False)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True,
                          collate_fn=collate_fn, persistent_workers=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True,
                          collate_fn=collate_fn, persistent_workers=True)

# v12.3: OneCycleLR — single ramp-up then long decay, no destructive restarts
total_steps = EPOCHS * len(train_loader) // ACCUM_STEPS
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=[LR * 0.3, LR],
    total_steps=total_steps,
    pct_start=WARMUP_PCT,
    anneal_strategy='cos',
    div_factor=25.0,       # initial_lr = max_lr / 25
    final_div_factor=1e4,  # final_lr = initial_lr / 1e4
)

print(f"Train  : {len(train_ds)} samples → {len(train_loader)} batches/ep")
print(f"Val    : {len(val_ds)} samples → {len(val_loader)} batches/ep")
print(f"OneCycleLR: {total_steps} optimizer steps, warmup {WARMUP_PCT*100:.0f}%")
print(f"Gradient accumulation: {ACCUM_STEPS} steps (effective batch {BATCH_SIZE*ACCUM_STEPS})")


def save_ckpt(path, ep, best, hist, sched_state=None):
    tmp = path.with_suffix(".tmp")
    torch.save({"model": model.state_dict(), "optimizer": optimizer.state_dict(),
                "scheduler": sched_state or scheduler.state_dict(),
                "epoch": ep, "best_chamfer": best, "history": hist,
                "config": {"version": "v12.3_spine_aware"}}, tmp)
    tmp.replace(path)
    print(f"  Saved → {path.name}  (ep {ep+1})")


def load_ckpt(path):
    st = torch.load(path, map_location=device, weights_only=False)
    model.load_state_dict(st["model"]); optimizer.load_state_dict(st["optimizer"])
    if st.get("scheduler"):
        try: scheduler.load_state_dict(st["scheduler"])
        except: print("  (scheduler state mismatch — reinitializing)")
    ep = st["epoch"] + 1; bc = st.get("best_chamfer", float("inf")); h = st.get("history", [])
    print(f"  Resumed from ep {ep} | best={bc:.3f} mm")
    return ep, bc, h


start_epoch, best_chamfer, history, no_improve = 0, float("inf"), [], 0
collapse_count = 0

if CKPT_PATH.exists():
    print(f"Found checkpoint: {CKPT_PATH.name}")
    start_epoch, best_chamfer, history = load_ckpt(CKPT_PATH)
else:
    print("No checkpoint — starting fresh.")

print(f"{'='*65}")
print(f"v12.3 training from ep {start_epoch+1}/{EPOCHS}")
print(f"Losses: chamfer+gap+axial+aux+extent | sw+proj+curvature")
print(f"OneCycleLR | accum={ACCUM_STEPS} | GRAD_CLIP={GRAD_CLIP}")
print(f"{'='*65}")


global_step = start_epoch * len(train_loader) // ACCUM_STEPS

for epoch in range(start_epoch, EPOCHS):

    # ── Encoder freeze/unfreeze ──
    if epoch < FREEZE_ENC_EPOCHS:
        for p in enc_params: p.requires_grad_(False)
        if epoch == start_epoch: print(f"  [Encoder FROZEN for ep 1-{FREEZE_ENC_EPOCHS}]")
    elif epoch == FREEZE_ENC_EPOCHS:
        for p in enc_params: p.requires_grad_(True)
        print(f"  [Encoder UNFROZEN at ep {epoch+1}]")

    if epoch == PHASE2_EPOCH:
        print(f"  [PHASE 2] Adding: sw, proj, curvature")

    # ── Training ──
    model.train(); t0 = time.time(); acc = {}; nb = 0; nan_count = 0
    optimizer.zero_grad(set_to_none=True)
    pbar = tqdm(enumerate(train_loader, 1), total=len(train_loader),
                desc=f'Ep {epoch+1:3d}/{EPOCHS}', leave=True, ncols=130)

    for bi, batch in pbar:
        batch = to_dev(batch)

        # Forward — fp32
        out  = model(batch["drr_ap"], batch["drr_lp"],
                     batch["P_ap"],   batch["P_lp"],
                     batch["center"], batch["scale"])

        loss, bd = total_loss(out, batch, epoch)
        loss = loss / ACCUM_STEPS  # scale for accumulation

        # NaN guard
        if torch.isnan(loss) or torch.isinf(loss):
            nan_count += 1
            if nan_count <= 10:
                nan_keys = [k for k, v in bd.items() if math.isnan(v) or math.isinf(v)]
                print(f"  ⚠ Loss NaN/Inf in batch {bi}. Bad: {nan_keys}")
            if nan_count > 50:
                print(f"  ⚠ Too many NaN batches ({nan_count}), stopping epoch"); break
            continue

        loss.backward()

        # Accumulation step
        if bi % ACCUM_STEPS == 0 or bi == len(train_loader):
            grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            if torch.isnan(grad_norm) or torch.isinf(grad_norm):
                nan_count += 1
                optimizer.zero_grad(set_to_none=True)
                continue
            optimizer.step()
            # Only step scheduler if not in freeze phase
            if epoch >= FREEZE_ENC_EPOCHS or True:
                scheduler.step()
            optimizer.zero_grad(set_to_none=True)
            global_step += 1

        for k, v in bd.items(): acc[k] = acc.get(k, 0) + float(v)
        nb += 1

        if bi % 50 == 0 or bi == len(train_loader):
            g = lambda k: acc.get(k, 0) / max(1, nb)
            pbar.set_postfix_str(
                f"ch={g('chamfer'):.3f} gap={g('gap'):.4f} ext={g('extent'):.4f} "
                f"nan={nan_count}")

    elapsed = time.time() - t0

    # ── Validation ──
    model.eval(); metrics = []; nd = 0
    with torch.no_grad():
        for batch in tqdm(val_loader, desc='  Val', leave=False, ncols=100):
            if nd >= MAX_EVAL: break
            batch = to_dev(batch)
            out = model(batch["drr_ap"], batch["drr_lp"],
                        batch["P_ap"],   batch["P_lp"],
                        batch["center"], batch["scale"])
            pw = out["pred_world"].cpu().numpy()
            gw = batch["gt_ppc_world"].cpu().numpy()
            for b in range(pw.shape[0]):
                if nd >= MAX_EVAL: break
                metrics.append(chamfer_np(pw[b], gw[b])); nd += 1

    if metrics:
        mc = np.mean([m["chamfer_mm"]   for m in metrics])
        f2 = np.mean([m["fscore_2mm"]   for m in metrics])
        hd = np.mean([m["hausdorff_95"] for m in metrics])
        g  = lambda k: acc.get(k, 0) / max(1, nb)

        phase = "P1" if epoch < PHASE2_EPOCH else "P2"

        print(f"\n[Ep {epoch+1} {phase}] {elapsed:.0f}s  Val: Chamfer={mc:.3f}mm  "
              f"F@2mm={f2:.3f}  HD95={hd:.3f}mm")
        print(f"  Train: ch={g('chamfer'):.3f} gap={g('gap'):.4f} "
              f"ax={g('axial'):.4f} ext={g('extent'):.4f} nan={nan_count}")
        if epoch >= PHASE2_EPOCH:
            print(f"  P2: sw={g('sw'):.4f} proj={g('proj'):.4f} "
                  f"curv={g('curvature'):.4f}")

        # Collapse detection
        if g("gap") >= 0.999:
            collapse_count += 1
            print(f"  ⚠ Collapse detected (gap≈1.0) — count {collapse_count}/5")
            if collapse_count >= 5 and BEST_CKPT.exists():
                print("  ⚠⚠ RESTORING best checkpoint, LR × 0.5")
                _, _, _ = load_ckpt(BEST_CKPT)
                for pg in optimizer.param_groups: pg["lr"] *= 0.5
                collapse_count = 0
                continue
        else:
            collapse_count = 0

        rec = {
            "epoch": epoch+1, "chamfer_mm": mc, "f2": f2, "hd95": hd,
            "train_gap": g("gap"), "train_axial": g("axial"),
            "train_extent": g("extent"), "train_chamfer": g("chamfer"),
            "nan_skips": nan_count, "phase": phase,
        }
        if epoch >= PHASE2_EPOCH:
            rec.update({"train_sw": g("sw"), "train_proj": g("proj"),
                        "train_curvature": g("curvature")})

        history.append(rec); append_log(TRAINING_LOG, rec)
        save_ckpt(CKPT_PATH, epoch, best_chamfer, history)

        if mc < best_chamfer:
            best_chamfer = mc; no_improve = 0
            save_ckpt(BEST_CKPT, epoch, best_chamfer, history)
            print(f"  ✓ New best: {best_chamfer:.3f} mm")
        else:
            no_improve += 1
            if no_improve >= EARLY_STOP_PATIENCE:
                print(f"  Early stop: {no_improve} epochs without improvement"); break

print(f"Done. Best val Chamfer: {best_chamfer:.3f} mm")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# TEST EVALUATION — v12.3: with Test-Time Augmentation (TTA)
# ══════════════════════════════════════════════════════════════════════════════

print("Test evaluation (with TTA: original + H-flip)...")
if BEST_CKPT.exists():
    st = torch.load(BEST_CKPT, map_location=device, weights_only=False)
    model.load_state_dict(st["model"])
    print(f"  Loaded best checkpoint from ep {st['epoch']+1} "
          f"(val Chamfer {st['best_chamfer']:.3f} mm)")
else:
    print("  WARNING: No best checkpoint — using current model state.")

model.eval()

# We need raw (un-normalized) images for TTA flip, so use a custom dataset
class TTALumbarDataset(Dataset):
    """Returns both original and H-flipped inputs for TTA."""
    def __init__(self, ids, root=DATA_ROOT):
        self.ids = ids; self.root = Path(root)
        self.norm = transforms.Normalize(mean=[0.485], std=[0.229])

    def __len__(self): return len(self.ids)

    def __getitem__(self, i):
        pid = self.ids[i]; d = self.root / pid
        dap = load_drr_image(d / "AP_0"  / "drr_AP_0.png")
        dlp = load_drr_image(d / "LP_90" / "drr_LP_90.png")
        Pap = load_projection_matrix(d / "AP_0"  / "P_AP_0.txt")
        Plp = load_projection_matrix(d / "LP_90" / "P_LP_90.txt")
        gt  = load_vtk_points(d / "gt_ppc.vtk")
        c   = gt.mean(0).astype(np.float32); s = compute_scale(gt)
        n   = len(gt)
        sel = np.random.choice(n, N_POINTS, replace=(n < N_POINTS))

        # Original
        dap_t = self.norm(torch.from_numpy(dap).unsqueeze(0).float())
        dlp_t = self.norm(torch.from_numpy(dlp).unsqueeze(0).float())

        # Flipped
        dap_f = dap[:, ::-1].copy()
        dlp_f = dlp[:, ::-1].copy()
        Pap_f = flip_P(Pap)
        Plp_f = flip_P(Plp)
        dap_ft = self.norm(torch.from_numpy(dap_f).unsqueeze(0).float())
        dlp_ft = self.norm(torch.from_numpy(dlp_f).unsqueeze(0).float())

        return {
            # Original
            "drr_ap": dap_t, "drr_lp": dlp_t,
            "P_ap": torch.from_numpy(Pap).float(),
            "P_lp": torch.from_numpy(Plp).float(),
            # Flipped
            "drr_ap_f": dap_ft, "drr_lp_f": dlp_ft,
            "P_ap_f": torch.from_numpy(Pap_f).float(),
            "P_lp_f": torch.from_numpy(Plp_f).float(),
            # GT
            "gt_ppc_world": torch.from_numpy(gt[sel]).float(),
            "center": torch.from_numpy(c).float(),
            "scale":  torch.from_numpy(s).float(),
            "patient_id": pid,
        }


test_loader = DataLoader(TTALumbarDataset(test_ids),
                         batch_size=2, shuffle=False, num_workers=NUM_WORKERS,
                         pin_memory=True, collate_fn=collate_fn)

all_m = []
all_m_no_tta = []

with torch.no_grad():
    for batch in tqdm(test_loader, desc='  Test+TTA', ncols=100):
        batch = to_dev(batch)
        B = batch["drr_ap"].shape[0]

        # Original pass
        out_orig = model(batch["drr_ap"], batch["drr_lp"],
                         batch["P_ap"],   batch["P_lp"],
                         batch["center"], batch["scale"])
        pw_orig = out_orig["pred_world"]

        # Flipped pass
        out_flip = model(batch["drr_ap_f"], batch["drr_lp_f"],
                         batch["P_ap_f"],   batch["P_lp_f"],
                         batch["center"],   batch["scale"])
        pw_flip = out_flip["pred_world"]

        # Average world-space predictions (TTA)
        pw_tta = 0.5 * (pw_orig + pw_flip)

        pw_tta_np = pw_tta.cpu().numpy()
        pw_orig_np = pw_orig.cpu().numpy()
        gt_np = batch["gt_ppc_world"].cpu().numpy()
        pids = batch["patient_id"]

        for b in range(B):
            m_tta = chamfer_np(pw_tta_np[b], gt_np[b])
            m_tta["patient_id"] = pids[b]
            all_m.append(m_tta)

            m_no = chamfer_np(pw_orig_np[b], gt_np[b])
            m_no["patient_id"] = pids[b]
            all_m_no_tta.append(m_no)

            save_vtk_points(pw_tta_np[b], RESULTS_DIR / f"{pids[b]}_pred.vtk")

# Print both for comparison
print(f"{'='*65}")
print(f"TEST RESULTS — NO TTA  ({len(all_m_no_tta)} patients)")
print(f"{'='*65}")
for key, lbl in [("chamfer_mm","Chamfer mm  "), ("fscore_1mm","F-score @1mm"),
                 ("fscore_2mm","F-score @2mm"), ("fscore_5mm","F-score @5mm"),
                 ("hausdorff_95","HD95 mm     ")]:
    vals = [m[key] for m in all_m_no_tta]
    print(f"  {lbl}  mean={np.mean(vals):.3f}  std={np.std(vals):.3f}  "
          f"median={np.median(vals):.3f}  p95={np.percentile(vals,95):.3f}")

print(f"\n{'='*65}")
print(f"TEST RESULTS — WITH TTA  ({len(all_m)} patients)")
print(f"{'='*65}")
for key, lbl in [("chamfer_mm","Chamfer mm  "), ("fscore_1mm","F-score @1mm"),
                 ("fscore_2mm","F-score @2mm"), ("fscore_5mm","F-score @5mm"),
                 ("hausdorff_95","HD95 mm     ")]:
    vals = [m[key] for m in all_m]
    print(f"  {lbl}  mean={np.mean(vals):.3f}  std={np.std(vals):.3f}  "
          f"median={np.median(vals):.3f}  p95={np.percentile(vals,95):.3f}")

csv_path = RESULTS_DIR / "test_results_v12_3_tta.csv"
with open(csv_path, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=["patient_id","chamfer_mm",
                                      "fscore_1mm","fscore_2mm","fscore_5mm","hausdorff_95"])
    w.writeheader(); w.writerows(all_m)
print(f"\nCSV → {csv_path}")

# Show improvement summary
c_no  = np.mean([m["chamfer_mm"] for m in all_m_no_tta])
c_tta = np.mean([m["chamfer_mm"] for m in all_m])
print(f"\nTTA improvement: {c_no:.3f} → {c_tta:.3f} mm "
      f"(Δ = {c_no - c_tta:.3f} mm)")
